# A1 corregido: Regresión logística (correcciones i, iii y iv)

Correr con este notebook en la misma carpeta que `metadata.csv`, `taxonomy_relabd.species.csv`,
`comun.py` y `pliegues_por_paciente.csv`.

- **(i)** Etiqueta binaria `was_preterm` y los mismos pliegues por paciente que A2 y A3. La probabilidad de cada
  paciente es el promedio de sus muestras.
- **(iii)** El umbral (Youden) se calcula solo con las pacientes de entrenamiento (`umbral_interno`).
- **(iv)** Se ajustan dos regresiones logísticas con el mismo modelo y los mismos pliegues:
  1. **raza**: la categoría racial en *one-hot* (una columna 0/1 por categoría, sin orden arbitrario);
  2. **microbioma**: las mismas abundancias por especie que ve el Random Forest (A2). Antes del modelo se
     quitan las especies presentes en menos del 5 % de las muestras de entrenamiento, se toma log10 y se
     estandariza. Así se puede comparar un modelo lineal con el Random Forest sobre la misma información.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from comun import (RESULTADOS, cargar_pliegues, iterar_pliegues, promedio_por_paciente,
                   umbral_interno, metricas)
RESULTADOS.mkdir(exist_ok=True)

In [ ]:
metadata = pd.read_csv("metadata.csv")
taxonomy = pd.read_csv("taxonomy_relabd.species.csv")

# Una fila por muestra, con su paciente, su etiqueta y sus especies (igual que A2)
data = pd.merge(metadata[['specimen', 'participant_id', 'was_preterm', 'NIH Racial Category']],
                taxonomy, on='specimen')

target = data['was_preterm'].astype(int).values
pacientes = data['participant_id'].values

# Modelo 1: raza en one-hot
feature_race = pd.get_dummies(data['NIH Racial Category'].fillna("Unknown")).astype(float).values

# Modelo 2: abundancias relativas por especie
feature_taxa = data.drop(columns=['specimen', 'participant_id', 'was_preterm', 'NIH Racial Category']).values

print(data.shape, '| pacientes:', len(set(pacientes)), '| muestras pretérmino:', target.mean().round(3))
print('columnas raza:', feature_race.shape[1], '| especies:', feature_taxa.shape[1])

In [ ]:
def entrenar_raza(X_train, y_train, X_test):
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]


def entrenar_microbioma(X_train, y_train, X_test):
    # Especies presentes en al menos el 5 % de las muestras de ENTRENAMIENTO
    especies = (X_train > 0).mean(axis=0) >= 0.05
    # log10 de la abundancia relativa (1e-6 evita log(0); el mínimo no nulo es ~1.6e-6)
    X_train = np.log10(X_train[:, especies] + 1e-6)
    X_test = np.log10(X_test[:, especies] + 1e-6)
    # Estandarizar con media y desviación del entrenamiento
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    model = LogisticRegression(max_iter=5000)
    model.fit(X_train, y_train)
    return model.predict_proba(X_test)[:, 1]


modelos = {
    'Regresión logística (raza)': (feature_race, entrenar_raza),
    'Regresión logística (microbioma)': (feature_taxa, entrenar_microbioma),
}

In [ ]:
pliegues = cargar_pliegues()
filas, predicciones = [], []
for rep, fold, ids_tr, ids_te in iterar_pliegues(pliegues):
    tr, te = np.isin(pacientes, list(ids_tr)), np.isin(pacientes, list(ids_te))
    for nombre, (X, entrenar) in modelos.items():
        # Umbral con datos de entrenamiento solamente (punto iii)
        umbral = umbral_interno(entrenar, X[tr], target[tr], pacientes[tr], semilla=100 + 10 * rep + fold)
        prob_muestra = entrenar(X[tr], target[tr], X[te])
        pp = promedio_por_paciente(pacientes[te], target[te], prob_muestra)
        filas.append(dict(modelo=nombre, rep=rep, fold=fold, **metricas(pp.y, pp.prob, umbral)))
        predicciones.append(pp.assign(modelo=nombre, rep=rep, fold=fold))
    print(f'rep {rep} fold {fold} listo')

pd.concat(predicciones).to_csv(RESULTADOS / 'pred_A1.csv', index=False)
res = pd.DataFrame(filas)
res.to_csv(RESULTADOS / 'metricas_A1.csv', index=False)
res.groupby('modelo')[['auc_roc', 'auc_pr', 'accuracy', 'precision', 'recall', 'f1', 'umbral']].agg(['mean', 'std']).round(3).T